# Text-to-Music Generation with Meta MusicGen\n\nThis practical uses the pretrained `facebook/musicgen-small` model for inference only. No dataset upload or model training is needed. For faster generation, select **Runtime > Change runtime type > T4 GPU** before running the cells. The first model load downloads several gigabytes from Hugging Face.\n

In [ ]:
import sys, torch\nprint('Python:', sys.version)\nprint('PyTorch:', torch.__version__)\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU:', torch.cuda.get_device_name(0))\nelse:\n    print('No GPU detected; generation can run on CPU but will be much slower.')

In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path\nfrom config import DEFAULT_CONFIG\nfrom music_generator import MusicGenerator\nOUTPUT_DIR = Path('/content/musicgen_outputs')\nmusicgen = MusicGenerator()\nmusicgen.load()  # Downloads and caches facebook/musicgen-small once.\nprint('Loaded model:', DEFAULT_CONFIG.model_id, '| device:', musicgen.device)

In [ ]:
prompt = 'Peaceful instrumental piano with soft flute and a relaxing atmosphere.'\nresult = musicgen.generate(prompt, duration_seconds=8, output_dir=OUTPUT_DIR, filename='musicgen_output.wav')\nprint('Prompt:', result.prompt)\nprint('Waveform shape:', result.waveform.shape, '| sample rate:', result.sampling_rate, '| seconds:', result.duration_seconds)\nprint('Saved WAV:', result.wav_path)

In [ ]:
from IPython.display import Audio, display\ndisplay(Audio(result.waveform, rate=result.sampling_rate))

In [ ]:
from scipy.io import wavfile\nrate, pcm = wavfile.read(result.wav_path)\nassert result.waveform.size > 0 and pcm.size > 0\nassert rate == result.sampling_rate\nprint('WAV verified:', rate, 'Hz,', pcm.dtype, pcm.shape)

In [ ]:
from google.colab import files\nfiles.download(str(result.wav_path))

## Optional: save a copy in Google Drive\nRun this cell only when persistent Drive storage is useful.\n

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\nDRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/MusicGen')\ndrive_result = musicgen.generate(prompt, duration_seconds=8, output_dir=DRIVE_OUTPUT_DIR, filename='musicgen_output.wav')\nprint('Saved to Drive:', drive_result.wav_path)

In [ ]:
prompts = [\n    'Energetic electronic music with a fast tempo and powerful synthesizers.',\n    'Indian classical instrumental music featuring sitar and tabla.',\n    'Cinematic orchestral music with emotional strings and dramatic percussion.',\n    'Gentle acoustic guitar music for studying.',\n]\nfrom IPython.display import Audio, display\ncomparison_results = []\nfor index, text in enumerate(prompts, start=1):\n    generated = musicgen.generate(text, duration_seconds=5, output_dir=OUTPUT_DIR, filename=f'prompt_{index}.wav')\n    comparison_results.append(generated)\n    print(f'Prompt {index}: {generated.prompt}')\n    display(Audio(generated.waveform, rate=generated.sampling_rate))

## Optional Gradio interface\nUpload or open this project in Colab so `app.py`, `config.py`, `music_generator.py`, and `requirements.txt` are in the current directory. The interface creates a share link; use one only if you intend it to be public.\n

In [ ]:
!python app.py --share